# GrowBot — Domain-Randomized SPIN-IN-PLACE Policy v2 (RMA-style, non-privileged)

Fixes for the v1 run (`logs_growbot_spin_olie_DR_RMA_train`, 77M steps). **The v1 policy did not spin.** A deterministic rollout of its final checkpoint nets `yaw rate ≈ 0.0004 rad/s` (basically zero) — it parks on its battery-heavy tail at ~-23° pitch and sits. The TensorBoard curves looked like progress (`YawRate_radps` 0.003→0.05) only because `ppo.train()` was never given `deterministic_eval=True`, so **every eval metric was measured under exploration noise, not the policy that deploys**. A stochastic rollout of the *same* checkpoint nets `0.22 rad/s` — almost all of the "learning" in the v1 logs was action-sampling jitter, not a paddling gait.

### What changed vs. v1
1. **`deterministic_eval=True`** on every `ppo.train()` call — the single biggest fix. Eval curves now measure the exact policy that ships.
2. **Reward rebalanced so spin dominates, not penalties.** v1's final per-episode decomposition was `spin +107` vs. `tilt −653, drift −309` — penalties outweighed the objective 9:1. v2 uses:
   * a **saturating spin term** `SPIN_SCALE * tanh(yaw_rate / YAW_SAT)` (bounded, can't be swamped by scale mismatches),
   * a small **alive bonus** so the return has a positive baseline instead of a −900 floor PPO has to claw out of,
   * **roll-only** tilt penalty at 1/7th the old weight (v1 penalized `|pitch|+|roll|` densely, but AbsPitch pinned at a *constant* ~28° from step 0 — that's the body's resting posture, not a controllable quantity, so most of the old tilt cost was an unavoidable fixed bias with no gradient),
   * a **deadbanded** drift penalty (free 15 cm circle to paddle in — yaw on two fore/aft hinges necessarily drags the body a little; taxing every cm was fighting the only gait that can turn),
   * a real **fall termination** (episode ends when `upright_cos` drops too low) instead of a dense per-step tilt tax.
3. **`ENTROPY_COST` 0.05 → 0.12.** v1's `ActionSaturation` was `0.000` for all 77M steps — the policy never once tried a large asymmetric leg swing, and `EntropyLoss` collapsed toward 0 early. On two hinges, meaningful yaw needs large actions; exploration needs to be pushed harder to find that gait before penalties clamp it down.
4. **Two-phase curriculum.** Phase 1 (`PHASE1_STEPS`) trains with drift/tilt penalties **off** — spin + alive bonus only — so the policy is free to discover a paddling gait. Phase 2 restores those params and fine-tunes with full penalties on, teaching it to hold position. brax 0.12.1's `ppo.train()` has no in-memory params-restore kwarg, only `restore_checkpoint_path` (an on-disk orbax checkpoint) — the phase-2 cell writes phase 1's params into that format itself before calling `ppo.train()`, so the handoff is still a single automatic step, just via disk instead of memory.

Everything else — DR ranges, actuator/cloud-relay model, 25 Hz control, the non-privileged `obs['state']` / privileged `obs['privileged']` split, network sizes — is **unchanged from v1 and the walk run**, so the deploy contract is still identical.

**Just press `Runtime → Run all`.**

## 1 · Install a pinned, mutually-compatible stack
The latest `brax` still calls a JAX API that the latest `jax` removed, so a fresh `pip install brax jax` breaks. These exact pins are verified to work together (and keep the asymmetric actor-critic API).

In [ ]:
# GPU build of the pinned stack (falls back to CPU automatically if no GPU).
!pip install -q "jax[cuda12]==0.4.36" "jaxlib==0.4.36" \
  "brax==0.12.1" "flax==0.10.2" "optax==0.2.4" "orbax-checkpoint==0.6.4" \
  "mujoco==3.2.7" "mujoco-mjx==3.2.7" "tensorboardX" 2>&1 | tail -3

# Load TensorBoard now so it's ready to watch training live.
%load_ext tensorboard

In [ ]:
import jax
print("JAX version:", jax.__version__)
if not jax.__version__.startswith("0.4.36"):
     print("\n[!] Wrong JAX version loaded (Colab pre-imported an old one).")
     print("    Do: Runtime > Restart session, then Runtime > Run all again.")
print("JAX devices:", jax.devices())
if jax.devices()[0].platform != "gpu":
    print("\n[!] No GPU detected — training will be SLOW.")
    print("    Set Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.")
else:
    print("\n[OK] GPU ready.")

## 2 · The robot model (MJCF)
The **olie enclosure body** (tub + backplate + leg), identical to `growbot_olie_body.xml` and unchanged from v1/the walk run. Domain randomization rescales/reweights it per-episode at runtime; this string is just the baseline.

In [ ]:
MJCF_XML = r"""
<mujoco model="Growbot">
  <option gravity="0 0 -9.81" timestep="0.005" integrator="RK4" solver="CG" iterations="10" ls_iterations="10"/>

  <default>
    <geom friction="1.2 0.1 0.1" solref="0.005 1" solimp="0.99 0.99 0.01" condim="3"/>
    <joint damping="0.03" armature="0.002"/>
    <position kp="0.75" kv="0.05" ctrlrange="-1.57 1.57" forcerange="-1.5 1.5"/>
  </default>

  <worldbody>
    <light name="sun" pos="0 0 3" dir="0 0 -1" diffuse="0.8 0.8 0.8" specular="0.2 0.2 0.2" castshadow="true"/>
    <geom name="floor" type="plane" size="0 0 0.05" rgba="0.76 0.87 0.70 1"/>

    <body name="base_body" pos="0 0 0.3">
      <joint name="root_joint" type="free"/>
      <!-- CoM shifted rear+low: 280g of the 427.254g torso mass is the battery, flush against
           the rear wall (-X, "butt") and underside (-Z), per user-specified footprint/position.
           fullinertia used (not diaginertia) because the rear+low offset couples X and Z (Ixz != 0). -->
      <inertial pos="-0.030801 0 -0.003015" mass="0.427254" fullinertia="0.00020856 0.00068954 0.00086672 0 -0.00002086 0"/>
      <geom name="torso_geom" type="box" size="0.084 0.04 0.0136" pos="0 0 0" rgba="0.7 0.7 0.7 1"/>
      <geom name="battery_visual" type="box" size="0.035 0.035 0.007" pos="-0.047 0 -0.0046" contype="0" conaffinity="0" rgba="0.15 0.15 0.15 1"/>

      <body name="right_leg" pos="0.0 -0.0475 0.009325">
        <joint name="joint_1" type="hinge" axis="0 1 0" limited="true" range="-90 90"/>
        <geom name="lower_leg_1" type="box" size="0.0105 0.0065 0.0369998" pos="0 0 -0.0369998" mass="0.0263731" rgba="0.2 0.2 0.8 1"/>
      </body>

      <body name="left_leg" pos="0.0 0.0475 0.009325">
        <joint name="joint_2" type="hinge" axis="0 1 0" limited="true" range="-90 90"/>
        <geom name="lower_leg_2" type="box" size="0.0105 0.0065 0.0369998" pos="0 0 -0.0369998" mass="0.0263731" rgba="0.2 0.2 0.8 1"/>
      </body>
    </body>
  </worldbody>

  <actuator>
    <position name="servo_1" joint="joint_1"/>
    <position name="servo_2" joint="joint_2"/>
  </actuator>
</mujoco>
"""

## 3 · Configuration — all the knobs in one place
DR ranges and the base PPO hyper-parameters are **identical to v1 / the walk run**. What's new:

* `ENTROPY_COST` raised 0.05 → 0.12 (v1 explored too little — `ActionSaturation` was 0.000 the entire run).
* The spin-reward knobs are reworked: `SPIN_SCALE`/`YAW_SAT` (saturating spin term), `ALIVE_BONUS` (positive baseline), `DRIFT_COST`/`DRIFT_DEADBAND` (free 15 cm to paddle in), `ROLL_TILT_COST` (roll only, not pitch — pitch is the body's fixed resting angle), `FALL_UPRIGHT_THRESH` (real termination instead of a dense tilt tax).
* `PHASE1_STEPS` / `PHASE2_STEPS` split `NUM_TIMESTEPS` into the curriculum: phase 1 trains spin-only (no drift/tilt cost) to find the gait, phase 2 restores those weights and fine-tunes with full penalties to hold position. `NUM_EVALS` is split proportionally (20 / 40) so eval density matches v1.

In [ ]:
HISTORY_LEN = 10          # frames of proprio+action stacked for the (non-privileged) policy obs
FRAME_DIM = 8             # per frame: roll,pitch,yaw,gyro(3),last_action(2)
CTRL_HZ_NFRAMES = 8       # 25 Hz control (timestep 0.005 * 8 = 0.04s)

# ---- domain randomization ranges (IDENTICAL to v1 / the walk run) ----
MASS_SCALE = (0.80, 1.25)         # alkaline vs lithium AA, phone weight
DCOM_X = (-0.030, 0.030)          # battery fore/aft position tolerance (m)
DCOM_Y = (-0.015, 0.015)
DCOM_Z = (-0.010, 0.015)          # battery low vs phone high
LEG_SHARED = (0.85, 1.15)         # print/material length variation (both legs together)
LEG_ASYM = (0.96, 1.04)           # per-leg assembly/horn-seat asymmetry
GAIN_MULT = (0.75, 1.25)          # servo Kp/Kv spread
FRICTION = (0.6, 1.4)
SLEW = (8.7, 10.5)                # rad/s
TAU = (0.015, 0.035)              # actuator lag (s)
CLOUD_DELAY = (0.020, 0.090)      # phone->cloud->Pico relay (s)
IMU_OFFSET = 0.26                 # +/- rad IMU mount misalignment
STALL_PROB = 0.08                 # cloud packet-drop probability

# ---- spin-in-place reward knobs (v2 — rebalanced so spin dominates penalties) ----
SPIN_DIR           = +1.0   # +1 = CCW (yaw-left), -1 = CW (yaw-right). Fixed per policy.
SPIN_SCALE          = 4.0   # max magnitude of the saturating spin term (was a linear 2.0x in v1)
YAW_SAT             = 1.0   # rad/s — tanh saturation scale; reward_spin = SPIN_SCALE*tanh(yaw_rate/YAW_SAT)
ALIVE_BONUS         = 0.4   # per-step bonus for being alive -> gives the return a positive baseline
DRIFT_COST          = 3.0   # penalty per metre of drift BEYOND the deadband (phase 2 only)
DRIFT_DEADBAND      = 0.15  # m — free radius to paddle within before drift is taxed
ROLL_TILT_COST      = 0.15  # penalty per rad of |roll| only (phase 2 only) — was 1.0 on |pitch|+|roll| in v1
FALL_UPRIGHT_THRESH = 0.5   # upright_cos below this -> episode terminates (replaces v1's dense pitch tax)

# ---- curriculum: phase 1 = spin-only warmup (finds the gait), phase 2 = full-penalty fine-tune ----
PHASE1_STEPS = 20_000_000   # ~1/3 of training: drift_cost=0, roll_tilt_cost=0 -> free to explore paddling
PHASE2_STEPS = 40_000_000   # ~2/3: restore phase-1 params, turn full penalties back on
NUM_EVALS_PHASE1 = 20
NUM_EVALS_PHASE2 = 40

# ---- PPO hyper-parameters (mirroring the successful DR_25hz / walk run) ----
NUM_TIMESTEPS       = PHASE1_STEPS + PHASE2_STEPS   # 60,000,000 total, same as v1's config
EPISODE_LENGTH      = 1000
NUM_ENVS            = 2048
BATCH_SIZE          = 1024
NUM_MINIBATCHES     = 32
NUM_UPDATES_PER_BATCH = 4
UNROLL_LENGTH       = 20
DISCOUNTING         = 0.99
LEARNING_RATE       = 3e-4
ENTROPY_COST        = 0.12         # v2: was 0.05 — v1 never once saturated an action (explored too little)
REWARD_SCALING      = 1.0
POLICY_HIDDEN       = (128, 128)   # non-privileged policy (reasons over the history)
VALUE_HIDDEN        = (256, 256)   # privileged critic
SEED                = 0

## 4 · Environment
`reset()` draws a fresh random body every episode and records the spawn XY (the "stay in place" anchor); `step()` applies the cloud-relay delay + servo slew/lag, steps the physics with that episode's randomized model, and emits a **dict observation**: `state` (non-privileged, for the policy) and `privileged` (latents, for the critic only) — all unchanged from v1.

**v2 reward/termination changes**, exposed as constructor kwargs so phase 1 and phase 2 can share one class with different weights:
* `reward = ALIVE_BONUS + SPIN_SCALE*tanh(SPIN_DIR*yaw_rate/YAW_SAT) - drift_cost*max(0, drift-drift_deadband) - roll_tilt_cost*|roll| - ctrl - action_rate`
* `done = 1` when `base_height` collapses **or** `upright_cos < fall_upright_thresh` — a real episode termination instead of v1's dense per-step tilt tax that penalized the body's fixed resting pitch every single step.
* Phase 1 env: `drift_cost=0, roll_tilt_cost=0` (spin-only). Phase 2 env: defaults (full penalties).

In [ ]:
import jax
from jax import numpy as jnp
import numpy as np
import mujoco
from mujoco import mjx
from brax import envs
from brax.envs.base import PipelineEnv, State
import functools

class GrowbotSpinEnv(PipelineEnv):
    """Spin-in-place variant. DR / actuator model / obs are identical to the walk env.
    Reward weights are constructor kwargs (defaults = full phase-2 penalties) so the same
    class serves both curriculum phases — see envs.get_environment(..., drift_cost=0, ...)
    in the training cells below."""
    def __init__(self, spin_scale=SPIN_SCALE, yaw_sat=YAW_SAT, drift_cost=DRIFT_COST,
                 drift_deadband=DRIFT_DEADBAND, roll_tilt_cost=ROLL_TILT_COST,
                 fall_upright_thresh=FALL_UPRIGHT_THRESH, **kwargs):
        mj_model = mujoco.MjModel.from_xml_string(MJCF_XML)
        mj_model.opt.solver = mujoco.mjtSolver.mjSOL_CG
        mj_model.opt.iterations = 6
        mj_model.opt.ls_iterations = 6

        # indices we need for domain randomization / rewards (static python ints)
        self._torso_bid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "base_body")
        self._legR_bid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "right_leg")
        self._legL_bid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "left_leg")
        self._legR_gid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "lower_leg_1")
        self._legL_gid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "lower_leg_2")

        sys = mjx.put_model(mj_model)
        super().__init__(sys=sys, backend="mjx", n_frames=CTRL_HZ_NFRAMES, **kwargs)

        # nominal leg geometry (half-length along local Z, and the box half-sizes)
        self._leg_half_len0 = float(sys.geom_size[self._legR_gid, 2])
        self._leg_hx = float(sys.geom_size[self._legR_gid, 0])   # thickness/2
        self._leg_hy = float(sys.geom_size[self._legR_gid, 1])   # width/2
        self._leg_mass0 = float(sys.body_mass[self._legR_bid])
        self._torso_ipos0 = np.array(sys.body_ipos[self._torso_bid])

        # v2 reward/termination weights (see class docstring)
        self._spin_scale = spin_scale
        self._yaw_sat = yaw_sat
        self._drift_cost = drift_cost
        self._drift_deadband = drift_deadband
        self._roll_tilt_cost = roll_tilt_cost
        self._fall_upright_thresh = fall_upright_thresh

    @property
    def action_size(self):
        return self.sys.nu

    # ---------------- domain randomization (IDENTICAL to v1 / the walk env) ----------------
    def _randomize(self, rng):
        keys = jax.random.split(rng, 13)
        mass_scale = jax.random.uniform(keys[0], (), minval=MASS_SCALE[0], maxval=MASS_SCALE[1])
        dcom = jnp.array([
            jax.random.uniform(keys[1], (), minval=DCOM_X[0], maxval=DCOM_X[1]),
            jax.random.uniform(keys[2], (), minval=DCOM_Y[0], maxval=DCOM_Y[1]),
            jax.random.uniform(keys[3], (), minval=DCOM_Z[0], maxval=DCOM_Z[1]),
        ])
        leg_shared = jax.random.uniform(keys[4], (), minval=LEG_SHARED[0], maxval=LEG_SHARED[1])
        legR_scale = leg_shared * jax.random.uniform(keys[5], (), minval=LEG_ASYM[0], maxval=LEG_ASYM[1])
        legL_scale = leg_shared * jax.random.uniform(keys[12], (), minval=LEG_ASYM[0], maxval=LEG_ASYM[1])
        gain_mult = jax.random.uniform(keys[6], (), minval=GAIN_MULT[0], maxval=GAIN_MULT[1])
        fric = jax.random.uniform(keys[7], (self.sys.ngeom,), minval=FRICTION[0], maxval=FRICTION[1])
        slew = jax.random.uniform(keys[8], (2,), minval=SLEW[0], maxval=SLEW[1])
        tau = jax.random.uniform(keys[9], (2,), minval=TAU[0], maxval=TAU[1])
        cloud_delay = jax.random.uniform(keys[10], (), minval=CLOUD_DELAY[0], maxval=CLOUD_DELAY[1])
        imu_offset = jax.random.uniform(keys[11], (3,), minval=-IMU_OFFSET, maxval=IMU_OFFSET)

        new_mass = self.sys.body_mass * mass_scale
        new_inertia = self.sys.body_inertia * mass_scale

        def scale_leg(sys_fields, gid, bid, s):
            gs, gp, bm, bi, bip = sys_fields
            half = self._leg_half_len0 * s
            gs = gs.at[gid, 2].set(half)
            gp = gp.at[gid, 2].set(-half)
            m = self._leg_mass0 * s * mass_scale
            bm = bm.at[bid].set(m)
            bip = bip.at[bid, 2].set(-half)
            a, b, c = 2 * self._leg_hx, 2 * self._leg_hy, 2 * half
            ixx = m / 12.0 * (b * b + c * c)
            iyy = m / 12.0 * (a * a + c * c)
            izz = m / 12.0 * (a * a + b * b)
            bi = bi.at[bid].set(jnp.array([ixx, iyy, izz]))
            return gs, gp, bm, bi, bip

        gs, gp = self.sys.geom_size, self.sys.geom_pos
        bm, bi, bip = new_mass, new_inertia, self.sys.body_ipos
        gs, gp, bm, bi, bip = scale_leg((gs, gp, bm, bi, bip), self._legR_gid, self._legR_bid, legR_scale)
        gs, gp, bm, bi, bip = scale_leg((gs, gp, bm, bi, bip), self._legL_gid, self._legL_bid, legL_scale)

        bip = bip.at[self._torso_bid].set(jnp.array(self._torso_ipos0) + dcom)

        new_gainprm = self.sys.actuator_gainprm * gain_mult
        new_biasprm = self.sys.actuator_biasprm * gain_mult
        new_friction = self.sys.geom_friction.at[:, 0].set(fric)

        rb = self.sys.geom_rbound
        rR = jnp.sqrt(self._leg_hx**2 + self._leg_hy**2 + (self._leg_half_len0 * legR_scale) ** 2)
        rL = jnp.sqrt(self._leg_hx**2 + self._leg_hy**2 + (self._leg_half_len0 * legL_scale) ** 2)
        rb = rb.at[self._legR_gid].set(rR).at[self._legL_gid].set(rL)

        rsys = self.sys.tree_replace({
            "body_mass": bm, "body_inertia": bi, "body_ipos": bip,
            "geom_size": gs, "geom_pos": gp, "geom_rbound": rb,
            "geom_friction": new_friction,
            "actuator_gainprm": new_gainprm, "actuator_biasprm": new_biasprm,
        })

        latents = {
            "mass_scale": mass_scale, "dcom": dcom,
            "legR_scale": legR_scale, "legL_scale": legL_scale,
            "gain_mult": gain_mult, "fric_mean": jnp.mean(fric),
            "slew": slew, "tau": tau, "cloud_delay": cloud_delay, "imu_offset": imu_offset,
        }
        return rsys, latents

    def reset(self, rng):
        rng, r1, r2, rdr = jax.random.split(rng, 4)
        rsys, lat = self._randomize(rdr)

        qpos = self.sys.qpos0 + jax.random.uniform(r1, (self.sys.nq,), minval=-0.05, maxval=0.05)
        qvel = jax.random.uniform(r2, (self.sys.nv,), minval=-0.05, maxval=0.05)
        data = self.pipeline_init(qpos, qvel)

        obs_hist = jnp.zeros((HISTORY_LEN, FRAME_DIM))
        action_hist = jnp.zeros((5, 2))
        delay_queue = jnp.zeros((5, 2))

        rng, orng = jax.random.split(rng)
        obs, obs_hist = self._get_obs(data, obs_hist, jnp.zeros(2), lat, orng)

        zero = jnp.zeros(())
        metrics = {k: zero for k in [
            "reward_spin", "reward_alive", "penalty_drift", "penalty_roll_tilt",
            "penalty_ctrl", "penalty_action_rate", "yaw_rate", "drift", "base_height",
            "upright", "abs_pitch", "abs_roll", "action_saturation", "fell"]}

        info = {
            "rng": rng, "sys": rsys, "obs_hist": obs_hist, "action_hist": action_hist,
            "real_motor_pos": jnp.zeros(2), "delay_queue": delay_queue,
            "last_action": jnp.zeros(2), "latents": lat,
            "spawn_xy": data.qpos[0:2],   # where the episode started -> "in place" anchor
        }
        return State(data, obs, zero, zero, metrics, info)

    def step(self, state, action):
        data0 = state.pipeline_state
        info = state.info
        sys = info["sys"]
        lat = info["latents"]

        # --- cloud relay: stall + delay (IDENTICAL to v1) ---
        rng, rstall = jax.random.split(info["rng"])
        is_stall = jax.random.uniform(rstall, ()) < STALL_PROB
        dq = info["delay_queue"]
        net_action = jnp.where(is_stall, dq[0], action)
        dq = jnp.concatenate([net_action[None], dq[:-1]])
        delay_idx = lat["cloud_delay"] / self.dt
        i0 = jnp.floor(delay_idx).astype(jnp.int32)
        rem = delay_idx - i0
        delayed = (1.0 - rem) * dq[i0] + rem * dq[i0 + 1]

        # --- actuator slew + 1-pole lag (IDENTICAL to v1) ---
        rmp = info["real_motor_pos"]
        max_delta = lat["slew"] * self.dt
        slewed = jnp.clip(delayed, rmp - max_delta, rmp + max_delta)
        alpha = self.dt / (lat["tau"] + self.dt)
        real_action = (1.0 - alpha) * rmp + alpha * slewed

        def phys(d, _):
            return self._pipeline.step(sys, d, real_action, self._debug), None
        data, _ = jax.lax.scan(phys, data0, (), self._n_frames)

        # --- orientation ---
        w, x, y, z = data.qpos[3], data.qpos[4], data.qpos[5], data.qpos[6]
        roll = jnp.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = jnp.arcsin(jnp.clip(2 * (w * y - z * x), -1, 1))
        upright = 1 - 2 * (x * x + y * y)
        base_h = data.qpos[2]

        # --- SPIN-IN-PLACE reward v2: saturating spin term + alive bonus + fall termination ---
        yaw_rate = data.qvel[5]   # body-frame gyro-z; the phone IMU's own channel
        reward_spin = self._spin_scale * jnp.tanh(SPIN_DIR * yaw_rate / self._yaw_sat)
        drift = jnp.linalg.norm(data.qpos[0:2] - info["spawn_xy"])
        penalty_drift = self._drift_cost * jnp.clip(drift - self._drift_deadband, 0.0, None)
        penalty_roll_tilt = self._roll_tilt_cost * jnp.abs(roll)
        ctrl_cost = 0.02 * jnp.sum(jnp.square(action))
        action_rate = 0.01 * jnp.sum(jnp.square(action - info["last_action"]))
        reward = ALIVE_BONUS + reward_spin - penalty_drift - penalty_roll_tilt - ctrl_cost - action_rate

        # --- diagnostics + real fall termination (replaces v1's dense per-step tilt tax) ---
        nominal_h = self._leg_half_len0 * 2 * 0.5 * (lat["legL_scale"] + lat["legR_scale"])
        fell_height = base_h < 0.4 * nominal_h
        fell_tilt = upright < self._fall_upright_thresh
        fell = jnp.float32(jnp.logical_or(fell_height, fell_tilt))
        sat = jnp.mean((jnp.abs(action) > 0.95 * 1.57).astype(jnp.float32))

        rng, orng = jax.random.split(rng)
        new_ahist = jnp.concatenate([action[None], info["action_hist"][:-1]])
        obs, new_obs_hist = self._get_obs(data, info["obs_hist"], action, lat, orng)

        state.metrics.update(
            reward_spin=reward_spin, reward_alive=jnp.asarray(ALIVE_BONUS),
            penalty_drift=penalty_drift, penalty_roll_tilt=penalty_roll_tilt,
            penalty_ctrl=ctrl_cost, penalty_action_rate=action_rate,
            yaw_rate=SPIN_DIR * yaw_rate, drift=drift, base_height=base_h, upright=upright,
            abs_pitch=jnp.abs(pitch), abs_roll=jnp.abs(roll),
            action_saturation=sat, fell=fell,
        )
        new_info = dict(info)
        new_info.update(rng=rng, action_hist=new_ahist, real_motor_pos=real_action,
                        delay_queue=dq, last_action=action, obs_hist=new_obs_hist)
        return state.replace(pipeline_state=data, obs=obs, reward=reward, done=fell, info=new_info)

    def _proprio_frame(self, data, last_action, lat, rng):
        w, x, y, z = data.qpos[3], data.qpos[4], data.qpos[5], data.qpos[6]
        roll = jnp.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = jnp.arcsin(jnp.clip(2 * (w * y - z * x), -1, 1))
        yaw = jnp.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))
        ang = jnp.array([roll, pitch, yaw]) + lat["imu_offset"]
        gyro = data.qvel[3:6]
        ra, rg = jax.random.split(rng)
        ang = ang + jax.random.normal(ra, (3,)) * 0.05
        gyro = gyro + jax.random.normal(rg, (3,)) * 0.1
        return jnp.concatenate([ang, gyro, last_action])

    def _get_obs(self, data, obs_hist, last_action, lat, rng):
        frame = self._proprio_frame(data, last_action, lat, rng)
        new_hist = jnp.concatenate([frame[None], obs_hist[:-1]])
        state_obs = new_hist.reshape(-1)

        w, x, y, z = data.qpos[3], data.qpos[4], data.qpos[5], data.qpos[6]
        roll = jnp.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = jnp.arcsin(jnp.clip(2 * (w * y - z * x), -1, 1))
        yaw = jnp.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))
        priv = jnp.concatenate([
            jnp.array([roll, pitch, yaw]), data.qvel[3:6], data.qvel[0:3],
            jnp.array([data.qpos[2]]),
            jnp.array([lat["mass_scale"]]), lat["dcom"],
            jnp.array([lat["legR_scale"], lat["legL_scale"], lat["gain_mult"], lat["fric_mean"]]),
            jnp.mean(lat["slew"], keepdims=True), jnp.mean(lat["tau"], keepdims=True),
            jnp.array([lat["cloud_delay"]]), lat["imu_offset"],
        ])
        return {"state": state_obs, "privileged": priv}, new_hist


envs.register_environment("growbot_spin", GrowbotSpinEnv)

## 5 · Sanity check (auto-runs)
Confirms both curriculum variants load (phase-1 spin-only weights and phase-2 full-penalty weights), the observation split is right, physics is finite, and `done` fires from `GrowbotSpinEnv`'s new fall termination — so a broken change fails here, loudly, before the multi-hour train.

In [ ]:
env_p2_probe = envs.get_environment("growbot_spin")  # defaults = full phase-2 penalties
env_p1_probe = envs.get_environment("growbot_spin", drift_cost=0.0, roll_tilt_cost=0.0)  # phase-1 weights

for label, e in [("phase-1 (spin-only)", env_p1_probe), ("phase-2 (full penalty)", env_p2_probe)]:
    _r = jax.jit(e.reset); _s = jax.jit(e.step)
    _st = _r(jax.random.PRNGKey(0))
    _st = _s(_st, jnp.zeros(e.action_size))
    assert bool(jnp.isfinite(_st.obs["state"]).all()), f"non-finite obs! ({label})"
    print(f"[{label}] policy obs: {_st.obs['state'].shape} | critic obs: {_st.obs['privileged'].shape} | "
          f"actions: {e.action_size} | reward={float(_st.reward):.3f} | done={float(_st.done):.0f}")

print("\nSanity check passed — starting setup for training.")

## 6 · Output dir + TensorBoard writer
Checkpoints + logs go to Google Drive if you allow it (survives disconnects → you can resume), otherwise to local `/content`. Checkpoints are split into `checkpoints/phase1/` and `checkpoints/phase2/` so the two curriculum stages don't collide. Open the TensorBoard panel below to watch training live — both phases log to the same continuous step axis.

In [ ]:
import os, datetime
from tensorboardX import SummaryWriter

RUN_NAME = "growbot_spin_v2_DR_RMA_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = f"/content/drive/MyDrive/Growbot/{RUN_NAME}"
except Exception as e:
    print("Drive not mounted (", e, ") -> using local /content (lost on disconnect).")
    BASE = f"/content/{RUN_NAME}"

LOG_DIR = f"{BASE}/logs"
CKPT_DIR_P1 = f"{BASE}/checkpoints/phase1"
CKPT_DIR_P2 = f"{BASE}/checkpoints/phase2"
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR_P1, exist_ok=True)
os.makedirs(CKPT_DIR_P2, exist_ok=True)
writer = SummaryWriter(LOG_DIR)
print("Logging to:", LOG_DIR)

In [ ]:
%tensorboard --logdir $LOG_DIR

## 7 · Logging & checkpoint callbacks
Brax reports each episode metric as a **sum over the episode's steps**, divided here by episode length for clean per-step means (yaw rate, mean drift, fall fraction…). Two v2-specific changes:

* `progress_fn` takes a **step offset** so phase 2's log points continue phase 1's x-axis instead of restarting at 0 — one continuous curve across both phases.
* Every `ppo.train()` call below passes **`deterministic_eval=True`**. This is the fix for the bug that made v1's curves meaningless: without it, brax evaluates with the *stochastic* (exploring) policy by default, and v1's "0.05 rad/s" yaw rate turned out to be almost entirely sampling noise — the deployed (deterministic) v1 policy actually spins at 0.0004 rad/s. `deterministic_eval=True` makes the curves match what a video rollout of the checkpoint would show.

In [ ]:
from brax.io import model
DT = None  # set once env is built below (both phases share the same dt)

SUM_TAGS = {
    "eval/episode_reward": "Reward/1_Total",
    "eval/episode_reward_spin": "Reward/2_Spin",
    "eval/episode_reward_alive": "Reward/3_Alive",
    "eval/episode_penalty_drift": "Penalty/Drift_total",
    "eval/episode_penalty_roll_tilt": "Penalty/RollTilt_total",
    "eval/episode_penalty_ctrl": "Penalty/Ctrl_total",
    "eval/episode_penalty_action_rate": "Penalty/ActionRate_total",
}
MEAN_TAGS = {
    "eval/episode_yaw_rate": "Diagnostics/YawRate_radps",
    "eval/episode_drift": "Diagnostics/Drift_m",
    "eval/episode_base_height": "Diagnostics/BaseHeight_m",
    "eval/episode_upright": "Diagnostics/Upright_cos",
    "eval/episode_abs_pitch": "Diagnostics/AbsPitch_rad",
    "eval/episode_abs_roll": "Diagnostics/AbsRoll_rad",
    "eval/episode_action_saturation": "Diagnostics/ActionSaturation",
    "eval/episode_fell": "Diagnostics/FallRate",
}
TRAIN_TAGS = {
    "eval/avg_episode_length": "Train/EvalEpisodeLen",   # now meaningful: fall = real termination
    "training/entropy_loss": "Train/EntropyLoss",
    "training/policy_loss": "Train/PolicyLoss",
    "training/v_loss": "Train/ValueLoss",
    "training/total_loss": "Train/TotalLoss",
    "training/sps": "Train/StepsPerSec",
}

def make_progress(step_offset, phase_label, ep_len_default):
    def progress(step, metrics):
        true_step = step + step_offset
        ep_len = float(metrics.get("eval/avg_episode_length", ep_len_default)) or ep_len_default
        logged = set()
        for k, tag in SUM_TAGS.items():
            if k in metrics: writer.add_scalar(tag, float(metrics[k]), true_step); logged.add(k)
        for k, tag in MEAN_TAGS.items():
            if k in metrics: writer.add_scalar(tag, float(metrics[k]) / ep_len, true_step); logged.add(k)
        for k, tag in TRAIN_TAGS.items():
            if k in metrics: writer.add_scalar(tag, float(metrics[k]), true_step); logged.add(k)
        if "eval/episode_yaw_rate" in metrics:
            total_yaw = float(metrics["eval/episode_yaw_rate"]) / ep_len * ep_len * DT  # = mean_rate*ep_len*DT
            writer.add_scalar("Diagnostics/YawTravelled_rad", total_yaw, true_step)
            writer.add_scalar("Diagnostics/Revolutions", total_yaw / (2 * np.pi), true_step)
        for k, v in metrics.items():
            if k in logged: continue
            try: val = float(v)
            except Exception: continue
            writer.add_scalar("Misc/" + k.replace("/", "_"), val, true_step)
        writer.flush()

        rew = float(metrics.get("eval/episode_reward", float("nan")))
        yr = float(metrics.get("eval/episode_yaw_rate", 0.0)) / ep_len
        drift = float(metrics.get("eval/episode_drift", 0.0)) / ep_len
        fell = float(metrics.get("eval/episode_fell", 0.0)) / ep_len
        sat = float(metrics.get("eval/episode_action_saturation", 0.0)) / ep_len
        if   fell > 0.5:                  note = "HIGH FALL RATE - toppling instead of spinning"
        elif yr < -0.05:                  note = "SPINNING THE WRONG WAY - flip SPIN_DIR or train longer"
        elif yr < 0.1 and fell < 0.2:     note = "stable but barely turning - too timid / underpowered / early"
        elif drift > 0.15 + DRIFT_DEADBAND: note = "WANDERING - spinning but drifting off the spot (raise DRIFT_COST)"
        elif sat > 0.7:                   note = "ACTIONS SATURATING - servo-limited across the mass range"
        else:                             note = "learning to spin in place"
        print(f"{phase_label} step {true_step:>10,} | reward {rew:8.1f} | yaw {yr:+.3f} rad/s | "
              f"drift {drift:.3f} m | fall {fell:4.2f} | sat {sat:4.2f} | ep_len {ep_len:5.0f} | {note}")
    return progress

def make_save_ckpt(ckpt_dir):
    def save_ckpt(step, make_policy, params):
        model.save_params(f"{ckpt_dir}/step_{step}.pkl", params)
    return save_ckpt

## 8 · Train — two-phase curriculum
Asymmetric actor-critic throughout: `policy_obs_key="state"` (non-privileged, deploys) vs `value_obs_key="privileged"` (latents, training only). **Phase 1** trains on the spin-only-reward environment (no drift/tilt cost) so exploration is free to find a paddling gait without being taxed for the drift and lean that gait necessarily causes. **Phase 2** restores phase 1's weights into the full-penalty environment (`envs.get_environment("growbot_spin")` with default costs) and fine-tunes it to hold position while still spinning. Both cells pass `deterministic_eval=True`. This is the multi-hour part — watch the TensorBoard above.

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=POLICY_HIDDEN,
    value_hidden_layer_sizes=VALUE_HIDDEN,
    policy_obs_key="state",       # non-privileged -> this is what deploys
    value_obs_key="privileged",   # latents -> critic only, discarded at deployment
)

env_phase1 = envs.get_environment("growbot_spin", drift_cost=0.0, roll_tilt_cost=0.0)
DT = float(env_phase1.dt)   # 0.04 s at 25 Hz -- shared by both phases

print("=== Phase 1: spin-only warmup (%s steps) ===" % f"{PHASE1_STEPS:,}")
print("Training… first eval appears after JIT compile (a few minutes).")
t0 = time.time()
make_inference_fn_p1, params_phase1, _ = ppo.train(
    environment=env_phase1,
    num_timesteps=PHASE1_STEPS,
    num_evals=NUM_EVALS_PHASE1,
    episode_length=EPISODE_LENGTH,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=UNROLL_LENGTH,
    num_minibatches=NUM_MINIBATCHES,
    num_updates_per_batch=NUM_UPDATES_PER_BATCH,
    discounting=DISCOUNTING,
    learning_rate=LEARNING_RATE,
    entropy_cost=ENTROPY_COST,
    reward_scaling=REWARD_SCALING,
    num_envs=NUM_ENVS,
    batch_size=BATCH_SIZE,
    network_factory=network_factory,
    progress_fn=make_progress(step_offset=0, phase_label="[phase1]", ep_len_default=EPISODE_LENGTH),
    policy_params_fn=make_save_ckpt(CKPT_DIR_P1),
    seed=SEED,
    deterministic_eval=True,   # v2 fix: eval the policy that actually deploys, not the exploring one
)
print(f"\nPhase 1 done in {(time.time()-t0)/60:.1f} min.")

## 8b · Phase 2 — restore + full-penalty fine-tune
Loads `params_phase1` into the full-penalty environment (`drift_cost`/`roll_tilt_cost` back to their configured values) and continues training. brax 0.12.1's `ppo.train()` has no in-memory `restore_params` kwarg, only `restore_checkpoint_path` (an on-disk orbax `PyTreeCheckpointer` checkpoint) — so this cell writes phase 1's params to `{BASE}/checkpoints/phase1_to_phase2_restore` in the exact `(normalizer_params, PPONetworkParams(policy, value))` structure brax expects, then points phase 2 at it. This is compatible because the network architecture, obs sizes, and action size are identical between phases — only the reward weights differ. Logs continue on the same TensorBoard step axis as phase 1 via `step_offset=PHASE1_STEPS`.

In [ ]:
env_phase2 = envs.get_environment("growbot_spin")  # defaults = full phase-2 penalties

# brax 0.12.1's ppo.train has no `restore_params` kwarg — only `restore_checkpoint_path`,
# an on-disk orbax PyTree checkpoint. Write phase 1's params into that exact structure
# (normalizer_params, PPONetworkParams(policy, value)) so phase 2 can load it.
import orbax.checkpoint as ocp
from brax.training.agents.ppo import losses as ppo_losses

RESTORE_CKPT_PATH = f"{BASE}/checkpoints/phase1_to_phase2_restore"
normalizer_params_p1, policy_params_p1, value_params_p1 = params_phase1
restore_target = (normalizer_params_p1,
                   ppo_losses.PPONetworkParams(policy=policy_params_p1, value=value_params_p1))
ocp.PyTreeCheckpointer().save(RESTORE_CKPT_PATH, restore_target, force=True)

print("=== Phase 2: full-penalty fine-tune (%s steps, resumed from phase 1) ===" % f"{PHASE2_STEPS:,}")
t0 = time.time()
make_inference_fn, params, _ = ppo.train(
    environment=env_phase2,
    num_timesteps=PHASE2_STEPS,
    num_evals=NUM_EVALS_PHASE2,
    episode_length=EPISODE_LENGTH,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=UNROLL_LENGTH,
    num_minibatches=NUM_MINIBATCHES,
    num_updates_per_batch=NUM_UPDATES_PER_BATCH,
    discounting=DISCOUNTING,
    learning_rate=LEARNING_RATE,
    entropy_cost=ENTROPY_COST,
    reward_scaling=REWARD_SCALING,
    num_envs=NUM_ENVS,
    batch_size=BATCH_SIZE,
    network_factory=network_factory,
    progress_fn=make_progress(step_offset=PHASE1_STEPS, phase_label="[phase2]", ep_len_default=EPISODE_LENGTH),
    policy_params_fn=make_save_ckpt(CKPT_DIR_P2),
    seed=SEED,
    restore_checkpoint_path=RESTORE_CKPT_PATH,
    deterministic_eval=True,
)
print(f"\nPhase 2 done in {(time.time()-t0)/60:.1f} min.")
print(f"Total training time: {(time.time()-t0)/60:.1f} min (phase 2 only; see phase 1 cell for its own timing).")

## 9 · Save the final policy
`params` (from phase 2) is the deployed policy. `params_phase1` is also saved for reference/debugging — e.g. to check in isolation whether the spin-only warmup found a real gait before the full-penalty fine-tune reshaped it.

In [ ]:
FINAL = f"{BASE}/growbot_spin_v2_DR_RMA_final.pkl"
model.save_params(FINAL, params)
PHASE1_SNAPSHOT = f"{BASE}/growbot_spin_v2_phase1_warmup.pkl"
model.save_params(PHASE1_SNAPSHOT, params_phase1)
print("Saved final (phase 2):", FINAL)
print("Saved phase-1 warmup snapshot:", PHASE1_SNAPSHOT)
print("\nDeploy note: the policy consumes ONLY obs['state'] (the non-privileged history) —")
print("identical layout to v1/the walk policy, so the browser-deploy obs_spec is unchanged.")
print("On the robot, build the same 10-frame stack of [roll,pitch,yaw, gyro_xyz, last_action(2)]")
print("from the phone IMU + last command, normalize with the saved obs stats, and run at 25 Hz.")
print("This policy spins", "CCW (yaw-left)" if SPIN_DIR > 0 else "CW (yaw-right)",
      "- flip SPIN_DIR and retrain for the other direction.")
print("\nBefore trusting these curves on hardware: render a clean DETERMINISTIC rollout of FINAL")
print("(e.g. with render_spin_ckpt_video.py) and confirm yaw rate matches Diagnostics/YawRate_radps —")
print("that agreement is exactly what v1 was missing.")

## How to read these logs

Every metric is averaged over **deterministic** evaluation episodes (v2 fix) spanning the full randomized body distribution, so the curves now reflect the policy that actually deploys — not exploration noise.

| TensorBoard tag | Healthy trend | If it goes wrong |
|---|---|---|
| `Reward/1_Total` | rises above 0 (alive bonus gives a positive floor) | stuck near/below 0 → spin term still losing to penalties |
| `Reward/2_Spin` | rises toward `SPIN_SCALE` (saturates at 4.0) | flat near 0 → not finding the gait; raise `ENTROPY_COST` or extend `PHASE1_STEPS` |
| `Reward/3_Alive` | ≈ `ALIVE_BONUS × EvalEpisodeLen`, drops if falling a lot | drops sharply → `FallRate` up, episodes ending early |
| `Diagnostics/YawRate_radps` | climbs to a steady positive value | ≈0 → **check this is v2, not a repeat of the v1 bug** (confirm `deterministic_eval=True` was actually passed); **<0 → wrong direction**, flip `SPIN_DIR` |
| `Diagnostics/Revolutions` | grows (full turns per episode) | — |
| `Diagnostics/Drift_m` | settles near/under `DRIFT_DEADBAND` (0.15 m) | grows well past it → raise `DRIFT_COST` in phase 2 |
| `Diagnostics/FallRate` | drops toward 0 | stays high → phase-1 gait is too violent for phase-2 penalties to tame; lengthen phase 2 or soften the transition |
| `Train/EvalEpisodeLen` | **now meaningful** (real fall termination) — should climb toward `EPISODE_LENGTH` | stays low → falling early and often |
| `Diagnostics/ActionSaturation` | should be **> 0** at some point in phase 1 | still 0.000 throughout → exploration still isn't finding large actions; raise `ENTROPY_COST` further |
| `Diagnostics/AbsPitch_rad` | expect it to *stay* near the v1 baseline (~0.5 rad) — this is the body's resting angle, not a controlled quantity anymore | — |
| `Diagnostics/AbsRoll_rad` | small, the only tilt axis now penalized | large → leaning hard into the turn |
| `Train/ValueLoss` | decreases/stable | exploding → DR spread too hard for the critic; narrow ranges or lengthen training |

**Tuning loop:** if phase 1 ends with `ActionSaturation` still at 0 and `YawRate` near 0, the gait was never found — raise `ENTROPY_COST` or extend `PHASE1_STEPS` before touching phase 2 at all. If phase 1 finds a gait (`YawRate` clearly positive, `sat > 0`) but phase 2 kills it (`YawRate` collapses back toward 0 as `Drift_m`/`FallRate` improve), `DRIFT_COST` or `ROLL_TILT_COST` are too aggressive for phase 2 — lower them.

**Always sanity-check with a deterministic video rollout before trusting the curves** — that's exactly the failure mode this v2 exists to fix.

## Resuming after a disconnect
Checkpoints are in `…/checkpoints/phase1/step_*.pkl` and `…/checkpoints/phase2/step_*.pkl` on your Drive — these are plain `brax.io.model.save_params` pickles (three-tuples of `(normalizer_params, policy_params, value_params)`), not orbax checkpoints, so they can't be passed straight to `restore_checkpoint_path`. To resume phase 2 specifically: load one with `normalizer_params, policy_params, value_params = model.load_params(".../phase2/step_XXXX.pkl")`, write it to an orbax checkpoint the same way the phase-1→phase-2 handoff cell does (`ocp.PyTreeCheckpointer().save(path, (normalizer_params, ppo_losses.PPONetworkParams(policy=policy_params, value=value_params)), force=True)`), then call `ppo.train(..., restore_checkpoint_path=path, ...)` with `num_timesteps` reduced by however many steps already ran.

## Training the other direction
This trains one turn direction (`SPIN_DIR = +1` → CCW). For the opposite direction, set `SPIN_DIR = -1` in the config cell and rerun both phases — you get a second `.pkl`. Since there is no command input, each policy spins one way; deploy whichever the robot needs, or load both and switch.